# Tips 05：Excelの複数シートを集計

**業務の困りごと**：部署別シートを1枚ずつ開いて合計する。

**このTipsでできること**：`sheet_name=None` は全シートを辞書形式で読みます。シート名を「部署」列に加えてから結合すると、元の所属を失わずに集計できます。

**使用ライブラリ**：pandas, openpyxl

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

営業・製造・総務の3シートがあるExcelを作ります。各シートは同じ「項目」「金額」列です。

最後の `display()` で練習用データ全体を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外のPython環境向け
import pandas as pd
from pathlib import Path
base=Path('/content') if Path('/content').exists() else Path.cwd()
# 複数シートを持つExcelを作る
src=base/'tips05_部署別.xlsx'
# 部署ごとに別シートを書き込む
with pd.ExcelWriter(src) as writer:
    for dept,amount in [('営業',2000),('製造',1500),('総務',1000)]:
        pd.DataFrame({'項目':['備品','交通費'],'金額':[amount,500]}).to_excel(writer,sheet_name=dept,index=False)
print('シート一覧（練習用データ）:')
display(pd.DataFrame({'シート名':pd.ExcelFile(src).sheet_names}))
# 作成した営業・製造・総務の全シートを表示する
for sheet_name, sheet_data in pd.read_excel(src, sheet_name=None).items():
    print(f'【{sheet_name}シート】')
    display(sheet_data)


## 2. 自動処理

`sheet_name=None` は全シートを辞書形式で読みます。シート名を「部署」列に加えてから結合すると、元の所属を失わずに集計できます。

**結果を見るポイント**：明細は **6行**、部署別の合計は **6,000円** です。

保存した成果物を読み直して `display()` で表示します。画面の表とダウンロードしたファイルの内容を照らし合わせてください。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
# 全シートを「シート名→表」の辞書で受け取る
sheets=pd.read_excel(src,sheet_name=None)
# 部署列を追加した表をためる
parts=[]
for dept,frame in sheets.items():
    frame['部署'] = dept
    parts.append(frame)
# 各部署の明細を縦に結合する
all_rows=pd.concat(parts,ignore_index=True)
# 部署別の金額を合計する
summary=all_rows.groupby('部署',as_index=False)['金額'].sum()
out=base/'Tips05_全社集計.xlsx'
with pd.ExcelWriter(out) as writer:
    all_rows.to_excel(writer,'明細',index=False)
    summary.to_excel(writer,'部署別',index=False)
print(summary)

# 明細と部署別集計の両シートを確認する
for sheet_name, preview in pd.read_excel(out, sheet_name=None).items():
    print(f'【{sheet_name}】 {len(preview)} 行')
    display(preview)


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

明細は **6行**、部署別の合計は **6,000円** です。


In [ ]:
# 件数や合計が期待値と一致するか検査する
assert len(sheets)==3 and summary['金額'].sum()==6000
print('確認OK:',out)


## 4. 持ち帰り・練習

**練習問題**：「項目別」シートも作りましょう。



**実務で使う前に**：シートによって列名が異なる場合は、結合前に列名をそろえます。

出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
